# DistilBERT Classification on Gendered and Degendered Datasets

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    matthews_corrcoef, balanced_accuracy_score,
    cohen_kappa_score, jaccard_score, hamming_loss,
    confusion_matrix, roc_auc_score, average_precision_score
)
from datasets import Dataset
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments,
)

In [ ]:
# Tokenizer
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

def tokenize(example):
    return tokenizer(example["text"], truncation=True, padding="max_length", max_length=512)

# Metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    acc = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="macro")
    mcc = matthews_corrcoef(labels, preds)
    bal_acc = balanced_accuracy_score(labels, preds)
    kappa = cohen_kappa_score(labels, preds)
    jaccard = jaccard_score(labels, preds, average="macro")
    hamming = hamming_loss(labels, preds)
    cm = confusion_matrix(labels, preds)

    metrics = {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "mcc": mcc,
        "balanced_accuracy": bal_acc,
        "cohen_kappa": kappa,
        "jaccard": jaccard,
        "hamming_loss": hamming,
    }

    per_class = precision_recall_fscore_support(labels, preds, average=None)
    for i, label in enumerate(np.unique(labels)):
        metrics[f"{label}_precision"] = per_class[0][i]
        metrics[f"{label}_recall"] = per_class[1][i]
        metrics[f"{label}_f1"] = per_class[2][i]

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            metrics[f"cm_{i}{j}"] = cm[i, j]

    try:
        metrics["auc_roc"] = roc_auc_score(labels, preds, multi_class="ovr")
        metrics["auc_pr"] = average_precision_score(labels, preds)
    except:
        metrics["auc_roc"] = None
        metrics["auc_pr"] = None

    return metrics


## Gendered Dataset

In [ ]:
df_gendered = pd.read_csv("data/combined_letters_gendered.csv")[["full_text", "labels"]].dropna()

if df_gendered["labels"].dtype == object:
    le_gendered = LabelEncoder()
    df_gendered["labels"] = le_gendered.fit_transform(df_gendered["labels"])


In [ ]:
# Train-test split
X_train_gendered, X_test_gendered, y_train_gendered, y_test_gendered = train_test_split(
    df_gendered["full_text"],
    df_gendered["labels"],
    test_size=0.2,
    stratify=df_gendered["labels"],
    random_state=42,
)

In [ ]:
# Dataset preparation
train_dataset_gendered = Dataset.from_dict({
    "text": X_train_gendered.tolist(),
    "labels": y_train_gendered.tolist()
})
test_dataset_gendered = Dataset.from_dict({
    "text": X_test_gendered.tolist(),
    "labels": y_test_gendered.tolist()
})

train_dataset_gendered = train_dataset_gendered.map(tokenize, batched=True).remove_columns("text")
test_dataset_gendered = test_dataset_gendered.map(tokenize, batched=True).remove_columns("text")


In [ ]:
# Model
model_gendered = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(set(df_gendered["labels"]))
)

In [ ]:
# Training arguments
training_args_gendered = TrainingArguments(
    output_dir="./results_gendered",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    learning_rate=2e-5,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="no",
    load_best_model_at_end=False,
    report_to=None,
)

trainer_gendered = Trainer(
    model=model_gendered,
    args=training_args_gendered,
    train_dataset=train_dataset_gendered,
    eval_dataset=test_dataset_gendered,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)


In [ ]:
# Training
trainer_gendered.train()

In [ ]:
# Evaluation
trainer_gendered.evaluate(test_dataset_gendered)

## Degendered Dataset

In [ ]:
df_degendered = pd.read_csv("data/combined_letters_degendered.csv")[["full_text", "labels"]].dropna()

if df_degendered["labels"].dtype == object:
    le_degendered = LabelEncoder()
    df_degendered["labels"] = le_degendered.fit_transform(df_degendered["labels"])

In [ ]:
# Train-test split
X_train_degendered, X_test_degendered, y_train_degendered, y_test_degendered = train_test_split(
    df_degendered["full_text"],
    df_degendered["labels"],
    test_size=0.2,
    stratify=df_degendered["labels"],
    random_state=42
)


In [ ]:
# Dataset preparation
train_dataset_degendered = Dataset.from_dict({
    "text": X_train_degendered.tolist(),
    "labels": y_train_degendered.tolist()
})
test_dataset_degendered = Dataset.from_dict({
    "text": X_test_degendered.tolist(),
    "labels": y_test_degendered.tolist()
})

train_dataset_degendered = train_dataset_degendered.map(tokenize, batched=True).remove_columns("text")
test_dataset_degendered = test_dataset_degendered.map(tokenize, batched=True).remove_columns("text")

In [ ]:
# Model
model_degendered = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(set(df_degendered["labels"]))
)

In [ ]:
# Training arguments
training_args_degendered = TrainingArguments(
    output_dir="./results_degendered",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    learning_rate=2e-5,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="no",
    load_best_model_at_end=False,
    report_to=None,
)

trainer_degendered = Trainer(
    model=model_degendered,
    args=training_args_degendered,
    train_dataset=train_dataset_degendered,
    eval_dataset=test_dataset_degendered,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

In [ ]:
# Training
trainer_degendered.train()

In [ ]:
# Evaluation
trainer_degendered.evaluate(test_dataset_degendered)
